In [ ]:
# Use ChromaDB for storing some vectors
# see https://docs.trychroma.com/guides
# - embedding
# - similarity / distance function
# - metadata
# - collections

In [ ]:
import chromadb
# setup Chroma in-memory, for easy prototyping. Can add persistence easily!
client = chromadb.Client()

# client = chromadb.PersistentClient(path="/path/to/save/to")

In [ ]:
from chromadb.utils import embedding_functions
embed_fn = embedding_functions.DefaultEmbeddingFunction()
embed_fn

In [ ]:
inf_chunk_1 = """A vector database, vector store or vector search engine is a database that can store vectors 
(fixed-length lists of numbers) along with other data items. Vector databases typically implement one or more 
Approximate Nearest Neighbor algorithms,so that one can search the database with a query vector to retrieve the 
closest matching database records."""

inf_chunk_2 = """Vectors are mathematical representations of data in a high-dimensional space. In this space, 
each dimension corresponds to a feature of the data, with the number of dimensions ranging from a few hundred to 
tens of thousands, depending on the complexity of the data being represented. A vector's position in this space 
represents its characteristics. Words, phrases, or entire documents, as well as images, audio, and other types of data, 
can all be vectorized."""

inf_chunk_3 = """The Moon is Earth's only natural satellite. It orbits at an average distance of 384,400 km (238,900 mi), 
about 30 times the diameter of Earth. Tidal forces between Earth and the Moon have synchronized the Moon's orbital period 
(lunar month) with its rotation period (lunar day) at 29.5 Earth days, causing the same side of the Moon to always face Earth.  
"""

vectors = embed_fn([inf_chunk_1]) 
print(type(vectors))              # the embedding function returns a list of vectors  
print(type(vectors[0]))           # each vector is a numpy array
print(vectors[0].shape)           # of shape (384,) 

In [ ]:
############### Similarity between two vectors is measured by Cosine Similarity ##################
import numpy as np

vector_1 = embed_fn([inf_chunk_1])[0]
vector_2 = embed_fn([inf_chunk_2])[0]
vector_3 = embed_fn([inf_chunk_3])[0]

def cosine_similarity(v1, v2):
    dot_product = np.dot(v1, v2)
    return  dot_product / (np.linalg.norm(v1) * np.linalg.norm(v2))


print("length of vector 1 :", np.linalg.norm(vector_1))              # the embedding function we used produces normed vectors
print("length of vector 2 :", np.linalg.norm(vector_2))              # therefore dot product and cosine similarity are the same
print("<vector 1,vector 2>:", np.dot(vector_1, vector_2))
print("cosine_similarity  :", cosine_similarity(vector_1, vector_2))

# If your embedding function produces L2-normalized vectors (i.e., each vector has norm 1), 
# then Euclidean distance and cosine similarity are equivalent for ranking and nearest-neighbor search, 
# but they are not numerically identical.
print("Euclidian dist     :", np.linalg.norm(vector_1 - vector_2)) 

In [ ]:
vectors = [vector_1, vector_2, vector_3]

for i, v1 in enumerate(vectors):
    for j, v2 in enumerate(vectors):
        print(f"({i+1},{j+1})",cosine_similarity(v1, v2), np.linalg.norm(v1 - v2))

In [ ]:
# Create a collection
try:
    client.delete_collection(name="my_collection")
except Exception as e:
   print(e)
    
collection = client.create_collection(name="my_collection", 
                                      embedding_function=embed_fn, 
                                      metadata={"hnsw:space": "cosine"} # l2 is the default
                                     )
collection = client.get_collection(name="my_collection", embedding_function=embed_fn)

In [ ]:
# Add documents and their metadata into the collection
import uuid

ids = []
for i  in range(3):
    ids.append(str(uuid.uuid4()))

collection.add(
    documents=[inf_chunk_1, inf_chunk_2, inf_chunk_3],
    metadatas=[
        {"wikipedia": "Vector database", "url": "https://en.wikipedia.org/wiki/Vector_database"}, 
        {"wikipedia": "Vector database", "url": "https://en.wikipedia.org/wiki/Vector_database"}, 
        {"wikipedia": "Moon", "url": "https://en.wikipedia.org/wiki/Moon"}],
    ids=ids
)

In [ ]:
print(collection.count())

In [ ]:
# Retrieval Step
question = "Tell me something about the moon."
question_embeddings = embed_fn([question])


result = collection.query(
    query_embeddings=question_embeddings,
    n_results=2,
)

In [ ]:
result #distance = 1 - cosine_similarity !

In [ ]:
result = collection.query(
    query_embeddings=question_embeddings,
    n_results=2,
    where={"wikipedia": "Vector database"}
)

In [ ]:
result